```
[ User Input ]
       │
       ▼
┌──────────────┐      Queries Text Embeddings      ┌─────────────────┐
│ Vector Search│ ────────────────────────────────> │ Vector Database │
└──────────────┘                                   └─────────────────┘
       │ Returns Context Articles
       ▼
┌──────────────┐      Pipes prompt + history       ┌─────────────────┐
│ LangChain /  │ ────────────────────────────────> │ Fine-tuned LLM  │
│ Orchestrator │                                   │ (via Ollama API)│
└──────────────┘                                   └─────────────────┘
       │ Streams tokens back
       ▼
┌──────────────┐
│ Streamlit UI │ ──> (Sleek ChatGPT-like interface layout)
└──────────────┘
```

In [1]:
!pip install --upgrade keras keras-hub

# Access Token: KGAT_f18fedec26d5958217cab67f43a8c994

In [3]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import keras
import keras_hub

keras.mixed_precision.set_global_policy("mixed_bfloat16")

import numpy as np
import kagglehub

kagglehub.login()


In [ ]:
!pip install --upgrade pandas pyarrow

In [4]:
import pandas as pd

corpus_path = keras.utils.get_file(
    fname = "corpus.parquet",
    origin= ("https://huggingface.co/datasets/rag-datasets/rag-mini-bioasq/"
             "resolve/main/data/passages.parquet/part.0.parquet"),
)
dataset_path = keras.utils.get_file(
    fname = "train.parquet",
    origin= ("https://huggingface.co/datasets/rag-datasets/rag-mini-bioasq/"
             "resolve/main/data/test.parquet/part.0.parquet"),
)

corpus, ds = pd.read_parquet(corpus_path), pd.read_parquet(dataset_path)

In [5]:
corpus.head()

,passage
id,
9797,New data on viruses isolated from patients wit...
11906,We describe an improved method for detecting d...
16083,We have studied the effects of curare on respo...
23188,Kinetic and electrophoretic properties of 230-...
23469,Male Wistar specific-pathogen-free rats aged 2...


In [6]:
import ast

def get_context(ids):
    ids = ast.literal_eval(ids)
    passages = corpus.loc[ids]["passage"]
    passages = passages[passages != "nan"]
    return "\n".join(passages.tolist())

ds["context"] = ds["relevant_passage_ids"].apply(get_context)
ds.drop(columns = ["relevant_passage_ids"], inplace= True)

In [7]:
ds.head()

,question,answer,context
id,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...",The major gene for Hirschsprung disease (HSCR)...
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,Pituitary adenylate cyclase-activating polypep...
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein",Cell invasion through basement membrane is a s...
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,Splicing remains an incompletely understood pr...
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,Bone destruction is a common feature of inflam...


In [8]:
import json
data = []

for _, row in ds.iterrows():
    user_prompt = f"Context:\n{row['context']}\n\nQuestion:\n{row['question']}"
    assistant_response = row['answer']
    
    data.append({
        "prompts": user_prompt,
        "responses": assistant_response
    })

print("Sample item structure:")
print(json.dumps(data[0], indent=2))

Sample item structure:
{
  "prompts": "Context:\nThe major gene for Hirschsprung disease (HSCR) encodes the receptor tyrosine \nkinase RET. In a study of 690 European- and 192 Chinese-descent probands and \ntheir parents or controls, we demonstrate the ubiquity of a >4-fold \nsusceptibility from a C-->T allele (rs2435357: p = 3.9 x 10(-43) in European \nancestry; p = 1.1 x 10(-21) in Chinese samples) that probably arose once within \nthe intronic RET enhancer MCS+9.7. With in vitro assays, we now show that the T \nvariant disrupts a SOX10 binding site within MCS+9.7 that compromises RET \ntransactivation. The T allele, with a control frequency of 20%-30%/47% and case \nfrequency of 54%-62%/88% in European/Chinese-ancestry individuals, is involved \nin all forms of HSCR. It is marginally associated with proband gender (p = 0.13) \nand significantly so with length of aganglionosis (p = 7.6 x 10(-5)) and \nfamiliality (p = 6.2 x 10(-4)). The enhancer variant is more frequent in the \ncomm

In [9]:
formatted_dict = {
    "prompts": [item["prompts"] for item in data],
    "responses": [item["responses"] for item in data]
}

In [10]:
ds = tf.data.Dataset.from_tensor_slices(formatted_dict)

test_ds = ds.take(10)
remaining_ds = ds.skip(10)

remaining_ds = remaining_ds.shuffle(
    buffer_size= remaining_ds.cardinality(),
    seed=102
)

val_ds = remaining_ds.take(250)
train_ds = remaining_ds.skip(250)

val_ds = val_ds.batch(4)
train_ds = train_ds.batch(4)

I0000 00:00:1780458375.716018    4920 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79187 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0


In [12]:
for row in test_ds:
    print(row["prompts"].numpy())
    print(row["responses"].numpy())
    print("-" * 40)

b'Context:\nThe major gene for Hirschsprung disease (HSCR) encodes the receptor tyrosine \nkinase RET. In a study of 690 European- and 192 Chinese-descent probands and \ntheir parents or controls, we demonstrate the ubiquity of a >4-fold \nsusceptibility from a C-->T allele (rs2435357: p = 3.9 x 10(-43) in European \nancestry; p = 1.1 x 10(-21) in Chinese samples) that probably arose once within \nthe intronic RET enhancer MCS+9.7. With in vitro assays, we now show that the T \nvariant disrupts a SOX10 binding site within MCS+9.7 that compromises RET \ntransactivation. The T allele, with a control frequency of 20%-30%/47% and case \nfrequency of 54%-62%/88% in European/Chinese-ancestry individuals, is involved \nin all forms of HSCR. It is marginally associated with proband gender (p = 0.13) \nand significantly so with length of aganglionosis (p = 7.6 x 10(-5)) and \nfamiliality (p = 6.2 x 10(-4)). The enhancer variant is more frequent in the \ncommon forms of male, short-segment, and 

In [13]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_270m")
gemma_lm.summary(line_length=100)

normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                             ┃                                Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                       │                   Vocab size: 262,144 │
└──────────────────────────────────────────────────────────┴───────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)   │ (None, None)            │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ token_ids (InputLayer)      │ (None, None)            │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ gemma3_backbone             │ (None, None, 640)       │    268,098,176 │ padding_mask[0][0],     │
│ (Gemma3Backbone)            │                         │                │ token_ids[0][0]         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ token_embedding             │ (None, None, 262144)    │    167,772,160 │ gemma3_backbone[0][0]   │
│ (ReversibleEmbedding)       │                         │                │                         │
└─────────────────────────────┴─────────────────────────┴────────────────┴─────────────────────────┘

 Total params: 268,098,176 (1022.71 MB)

 Trainable params: 268,098,176 (1022.71 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
gemma_lm.backbone.enable_lora(rank=8)

In [15]:
gemma_lm.summary(line_length = 100)

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                             ┃                                Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                       │                   Vocab size: 262,144 │
└──────────────────────────────────────────────────────────┴───────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)   │ (None, None)            │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ token_ids (InputLayer)      │ (None, None)            │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ gemma3_backbone             │ (None, None, 640)       │    268,632,704 │ padding_mask[0][0],     │
│ (Gemma3Backbone)            │                         │                │ token_ids[0][0]         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ token_embedding             │ (None, None, 262144)    │    167,772,160 │ gemma3_backbone[0][0]   │
│ (ReversibleEmbedding)       │                         │                │                         │
└─────────────────────────────┴─────────────────────────┴────────────────┴─────────────────────────┘

 Total params: 268,632,704 (1.00 GB)

 Trainable params: 534,528 (2.04 MB)

 Non-trainable params: 268,098,176 (1022.71 MB)

In [18]:
ps = gemma_lm.preprocessor
ps.sequence_length = 2048
batch = next(iter(train_ds))
x, y, sample_weight = ps(batch)
print(x["token_ids"].shape, x["padding_mask"].shape, y.shape, sample_weight.shape)

(4, 2048) (4, 2048) (4, 2048) (4, 2048)


In [19]:
x["token_ids"][0, :5], y[0, :5]

(<tf.Tensor: shape=(5,), dtype=int32, numpy=array([     2,   3637, 236787,    109,  14977], dtype=int32)>,
 <tf.Tensor: shape=(5,), dtype=int32, numpy=array([  3637, 236787,    109,  14977, 236787], dtype=int32)>)

In [20]:
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(
    learning_rate=2e-5,
    weight_decay=0.01,
    clipnorm=1.0
),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

history = gemma_lm.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=3
)

Epoch 1/3


I0000 00:00:1780458610.889175   18182 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1115/1115 ━━━━━━━━━━━━━━━━━━━━ 316s 212ms/step - loss: 0.0588 - sparse_categorical_accuracy: 0.6207 - val_loss: 0.0600 - val_sparse_categorical_accuracy: 0.6075
Epoch 2/3
1115/1115 ━━━━━━━━━━━━━━━━━━━━ 158s 141ms/step - loss: 0.0535 - sparse_categorical_accuracy: 0.6395 - val_loss: 0.0519 - val_sparse_categorical_accuracy: 0.6333
Epoch 3/3
1115/1115 ━━━━━━━━━━━━━━━━━━━━ 157s 141ms/step - loss: 0.0527 - sparse_categorical_accuracy: 0.6427 - val_loss: 0.0511 - val_sparse_categorical_accuracy: 0.6441


In [22]:
gemma_lm.save_to_preset("./my_finetuned_gemma")

In [23]:
!nvidia-smi

Wed Jun  3 04:02:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   43C    P0             66W /  400W |   79702MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!kill -9 5802

In [24]:
prompt_fec75 = (
    "<start_of_turn>user\n"
    "Context:\n"
    "A 45-year-old female patient diagnosed with Stage II human epidermal growth factor "
    "receptor 2-negative (HER2-) breast cancer is scheduled to undergo adjuvant chemotherapy. "
    "The oncology team selects a standard 21-day cycling protocol. The regimen initiates with "
    "an intravenous infusion of Fluorouracil at 500 mg/m², followed immediately by a 75 mg/m² "
    "dose of Epirubicin to leverage its anthracycline-induced DNA intercalation mechanisms. "
    "The cycle concludes with an injection of the alkylating agent Cyclophosphamide at 500 mg/m². "
    "This specific combination will be repeated every 3 weeks for a total of 4 continuous cycles, "
    "requiring careful monitoring of cardiac ejection fractions due to cumulative anthracycline exposure.\n\n"
    "Question:\n"
    "Which three specific cytotoxic drugs comprise the FEC-75 chemotherapy regimen described in this patient's protocol?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

predicted_response = gemma_lm.generate(prompt_fec75, max_length=2000, strip_prompt=True)

print("--- MODEL PREDICTION ---")
print(predicted_response)

--- MODEL PREDICTION ---
The three specific cytotoxic drugs comprise the FEC-75 chemotherapy regimen described in this patient's protocol:
*   Fluorouracil (500 mg/m²)
*   Epirubicin (75 mg/m²)
*   Cyclophosphamide (500 mg/m²)<end_of_turn>


In [30]:
reloaded_model = keras_hub.models.Gemma3CausalLM.from_preset(
    "./my_finetuned_gemma",
    dtype="bfloat16",
)
test_ds = test_ds.batch(4)
print(reloaded_model.evaluate(test_ds))

3/3 ━━━━━━━━━━━━━━━━━━━━ 42s 8s/step - loss: 0.0411 - sparse_categorical_accuracy: 0.7608
[0.04105711728334427, 0.7607758641242981]


In [31]:
import shutil

# This zips './my_finetuned_gemma' into a file named 'my_finetuned_gemma.zip'
shutil.make_archive("my_finetuned_gemma", "zip", "./my_finetuned_gemma")
print("Folder successfully zipped!")

Folder successfully zipped!


In [35]:
prompt_lncRNA = (
    "<start_of_turn>user\n"
    "Context:\n"
    "Splicing remains an incompletely understood process. Recent findings suggest "
    "that chromatin structure participates in its regulation. Here, we analyze the "
    "RNA from subcellular fractions obtained through RNA-seq in the cell line K562. "
    "We show that in the human genome, splicing occurs predominantly during "
    "transcription. We introduce the coSI measure, based on RNA-seq reads mapping to "
    "exon junctions and borders, to assess the degree of splicing completion around "
    "internal exons. We show that, as expected, splicing is almost fully completed in "
    "cytosolic polyA+ RNA. ... "
    "The human genome contains many thousands of long noncoding RNAs (lncRNAs). "
    "Our analyses indicate that lncRNAs are generated through pathways similar to that "
    "of protein-coding genes, with similar histone-modification profiles, splicing "
    "signals, and exon/intron lengths. ...\n\n"
    "Question:\n"
    "What is coSI measure?<end_of_turn>\n"
    "<start_of_turn>model\n"
)
predicted_response = reloaded_model.generate(prompt_lncRNA, max_length=2000, strip_prompt=True)

print("--- MODEL PREDICTION ---")
print(predicted_response)

--- MODEL PREDICTION ---
CoSI measure is a measure of the degree of splicing completion around internal exons in the human genome.<end_of_turn>
